# NHS Patient Feedback Intelligence App
## 01 — Exploration and Modelling

**Author:** Yenlik Gaisina  
**Dataset:** NHS Choices Patient Comments and Ratings (data.gov.uk)  
**Purpose:** Portfolio project — end-to-end NLP pipeline from raw patient comments to deployed classifier

---

## 1. Business Problem

Healthcare organisations receive thousands of patient feedback comments every year. Reading and categorising them manually is slow, inconsistent, and resource-intensive.

**Business question:** Can we automatically classify NHS patient comments into service-improvement themes to help operational teams prioritise action faster?

**Output:** A prototype app that takes a patient comment and returns:
- Predicted service-improvement theme
- Sentiment (positive / negative / neutral)
- Confidence score
- Suggested operational action

**Important caveat:** This is a portfolio prototype for service-improvement triage only. It is not a clinical tool and must not be used for diagnosis, patient safety decisions, or automated complaint handling without human review.

## 2. Data Loading

In [ ]:
import pandas as pd
import numpy as np

hospital_path = "../data/raw/hospital_comments.xls"
gp_path = "../data/raw/gp_comments.xls"

hospital = pd.read_excel(hospital_path, engine="openpyxl")
gp = pd.read_excel(gp_path, engine="openpyxl")

print("Hospital shape:", hospital.shape)
print("GP shape:", gp.shape)

display(hospital.head())
display(gp.head())

print("\nHospital columns:")
print(hospital.columns.tolist())

print("\nGP columns:")
print(gp.columns.tolist())

In [ ]:
# ── STOP HERE after first run ──────────────────────────────────────────────
# Inspect the column names above and update these two variables before
# running the rest of the notebook.
# ──────────────────────────────────────────────────────────────────────────

hospital_text_col = "Comment"   # confirmed from real data inspection
gp_text_col       = "Comment"   # confirmed from real data inspection
rating_col        = "Rating"    # confirmed — Rating column present in both files

# Build clean combined dataframe
cols_h = [hospital_text_col]
cols_g = [gp_text_col]

if rating_col and rating_col in hospital.columns:
    cols_h.append(rating_col)
if rating_col and rating_col in gp.columns:
    cols_g.append(rating_col)

hospital_clean = hospital[cols_h].copy()
hospital_clean = hospital_clean.rename(columns={hospital_text_col: "comment"})
hospital_clean["service_type"] = "Hospital"

gp_clean = gp[cols_g].copy()
gp_clean = gp_clean.rename(columns={gp_text_col: "comment"})
gp_clean["service_type"] = "GP"

if rating_col:
    gp_clean = gp_clean.rename(columns={rating_col: rating_col})

df = pd.concat([hospital_clean, gp_clean], ignore_index=True)

df = df.dropna(subset=["comment"])
df["comment"] = df["comment"].astype(str)
df = df[df["comment"].str.strip().str.len() > 20]
df = df.reset_index(drop=True)

print("Combined shape:", df.shape)
display(df.head())

## 3. Data Cleaning

In [ ]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [
        word for word in tokens
        if word.isalpha()
        and word not in stop_words
        and len(word) > 2
    ]
    return " ".join(tokens)

df["clean_comment"] = df["comment"].apply(clean_text)
df = df[df["clean_comment"].str.len() > 0].copy()
df = df.reset_index(drop=True)

print(f"Rows after cleaning: {len(df):,}")
display(df[["comment", "clean_comment", "service_type"]].head(5))

## 4. Exploratory Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# --- Comment length distribution ---
df["comment_length"] = df["comment"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["comment_length"], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Comment length distribution (characters)")
axes[0].set_xlabel("Length")
axes[0].set_ylabel("Count")

df["service_type"].value_counts().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_title("Comments by service type")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print(df["comment_length"].describe())

In [ ]:
# --- Word cloud ---
all_text = " ".join(df["clean_comment"])

wc = WordCloud(
    width=900,
    height=400,
    background_color="white",
    max_words=150,
    colormap="Blues"
).generate(all_text)

plt.figure(figsize=(14, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Most frequent words in patient comments", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Theme Labelling

Because the dataset does not contain pre-existing theme labels, we create **rule-based starter labels** using keyword matching. These are not expert-annotated clinical categories — they are a starting point for supervised learning.

Themes:
- `appointment_access`
- `communication`
- `staff_attitude`
- `waiting_time`
- `treatment_quality`
- `administration`
- `facilities`
- `other`

In [ ]:
theme_keywords = {
    "appointment_access": [
        "appointment", "book", "booking", "access", "available",
        "phone", "call", "reception", "gp appointment"
    ],
    "communication": [
        "communication", "explain", "explained", "told", "listen",
        "listened", "information", "contact", "update"
    ],
    "staff_attitude": [
        "rude", "kind", "helpful", "unhelpful", "staff", "nurse",
        "doctor", "receptionist", "attitude", "care"
    ],
    "waiting_time": [
        "wait", "waiting", "delay", "delayed", "late", "queue",
        "hours", "minutes"
    ],
    "treatment_quality": [
        "treatment", "diagnosis", "diagnosed", "medicine", "medication",
        "pain", "care", "procedure", "symptoms"
    ],
    "administration": [
        "letter", "referral", "form", "admin", "paperwork",
        "record", "prescription", "test results", "results"
    ],
    "facilities": [
        "clean", "dirty", "parking", "room", "ward",
        "toilet", "building", "facility", "equipment"
    ]
}

def assign_theme(text):
    text = str(text).lower()
    scores = {
        theme: sum(1 for kw in keywords if kw in text)
        for theme, keywords in theme_keywords.items()
    }
    best_theme = max(scores, key=scores.get)
    return "other" if scores[best_theme] == 0 else best_theme

df["theme"] = df["comment"].apply(assign_theme)

print(df["theme"].value_counts())

ax = df["theme"].value_counts().plot(
    kind="barh", figsize=(10, 5), color="steelblue", edgecolor="white"
)
ax.set_title("Comment counts by theme (rule-based starter labels)")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()

## 5b. Sentiment Scoring

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.1:
        return "positive"
    elif polarity < -0.1:
        return "negative"
    return "neutral"

df["sentiment"] = df["comment"].apply(get_sentiment)

print(df["sentiment"].value_counts())

# Sentiment by theme
sentiment_theme = df.groupby(["theme", "sentiment"]).size().unstack(fill_value=0)
sentiment_theme.plot(kind="bar", figsize=(12, 5), colormap="Set2", edgecolor="white")
plt.title("Sentiment breakdown by theme")
plt.xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Baseline Model: TF-IDF + Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# Exclude "other" from training — it is a catch-all, not a real class
model_df = df[df["theme"] != "other"].copy()

X = model_df["comment"]
y = model_df["theme"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=10000
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average="macro"))

## 7. Model Evaluation

In [ ]:
print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred))

In [ ]:
import seaborn as sns

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
    cmap="Blues"
)
plt.title("Confusion matrix — TF-IDF + Logistic Regression")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
errors = model_df.loc[X_test.index].copy()
errors["predicted"] = y_pred
errors = errors[errors["theme"] != errors["predicted"]]

print(f"Total misclassified: {len(errors):,} of {len(X_test):,}")
print("\nMost common misclassification pairs:")
print(
    errors.groupby(["theme", "predicted"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

print("\nSample misclassified comments:")
display(
    errors[["comment", "theme", "predicted"]]
    .sample(min(10, len(errors)), random_state=42)
)

## 9. Save Model and Processed Data

In [ ]:
import joblib
from pathlib import Path

Path("../models").mkdir(exist_ok=True)
Path("../data/processed").mkdir(exist_ok=True)

joblib.dump(clf, "../models/model.pkl")
df.to_csv("../data/processed/nhs_patient_feedback_processed.csv", index=False)

print("Saved:")
print("  ../models/model.pkl")
print("  ../data/processed/nhs_patient_feedback_processed.csv")

In [ ]:
# --- MLflow experiment tracking ---
import mlflow
import mlflow.sklearn

mlflow.set_experiment("nhs_patient_feedback_theme_classifier")

with mlflow.start_run():
    mlflow.log_param("model_type", "TF-IDF + Logistic Regression")
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("macro_f1", f1_score(y_test, y_pred, average="macro"))
    mlflow.sklearn.log_model(clf, "model")

print("MLflow run logged.")

## 10. Responsible AI Notes

| Risk | Mitigation |
|------|------------|
| **Misclassification** | Show confidence score; recommend human review for low-confidence predictions |
| **Weak labels** | Labels are rule-based starter labels — document clearly and avoid clinical claims |
| **Sensitive patient content** | Do not publish raw identifiable comments in public demos |
| **Over-trust by users** | Visible disclaimer in the app: operational triage only, not clinical |
| **Bias** | Monitor performance across comment types; no automated action without human oversight |
| **Data staleness** | NHS Choices dataset covers a fixed period — plan for retraining if deploying |

Full documentation: see `reports/model_card.md` and `reports/responsible_ai_risk_assessment.md`.

## 11. Deployment Plan

1. **Local test:** `streamlit run app.py` — paste a comment, confirm predictions look sensible
2. **Run tests:** `pytest` — confirm all assertions pass
3. **GitHub:** push to repo — GitHub Actions will run tests automatically on every push
4. **Hugging Face Spaces:** upload `app.py`, `predict.py`, `requirements.txt`, `models/model.pkl` — choose Streamlit SDK
5. **README:** add the live Hugging Face Spaces link and update the Model Performance table with real metrics

**Future iterations:**
- Replace TextBlob with a Hugging Face sentiment model fine-tuned on healthcare text
- Fine-tune DistilBERT on expert-labelled NHS comments for the theme classifier
- Add confidence threshold routing: predictions below 0.6 confidence go to a human review queue
- Build a manager dashboard showing aggregate theme trends over time